In [39]:
# Importa base de produtos e novo arquivo

import pandas as pd

arqv = r"G:\Meu Drive\Python RPA\Projeto IA\TeleVendas\modelo - RN\ml-televendas\Interface\teste_televendas.xlsx"
qtd = r"C:\Users\Rafael\Downloads\base_quantidades.CSV"


In [48]:
# Classe validações


import math
import pandas as pd


class Validacoes():
    
    def __init__(self, arquivo_pedido, arquivo_qtd):
        self.arquivo_pedido = arquivo_pedido
        self.arquivo_qtd = arquivo_qtd


        self.qtd = None
        self.df_merge = None
        self.df = None


    def processa_df(self):

        self.df = pd.read_excel(self.arquivo_pedido)

        self.qtd = pd.read_csv(self.arquivo_qtd)


    def etl_qtd(self):
        colunas = ['="ean"', '="qt_embalagem_venda"']
        self.qtd = self.qtd[colunas]


    def join_colunas(self):
        self.df['Unnamed: 0'] = self.df['Unnamed: 0'].astype(str)
        self.qtd['="ean"'] = self.qtd['="ean"'].astype(str)
        # Faz um merge com a base de dados e a base de EAN da GAM.
        self.df_merge = pd.merge(self.df, self.qtd, left_on= 'Unnamed: 0', right_on = '="ean"')

        #Remove colunas que são toda NaN
        self.df_merge = self.df_merge.dropna(axis=1, how='all')

        print(self.df_merge.columns)

        colunas = [ 'Código', 'Descrição', 'Fabricante', 'Und','Embalagem','Qtd','Desc.','Vlr Unit','Vlr IPI','Vlr Total','Icms','Filial','Prazo','num_pedido','="qt_embalagem_venda"']

        self.df_merge.index = self.df_merge['Unnamed: 0.1']
        self.df_merge.drop('Unnamed: 0.1', axis=1, inplace=True)
        self.df_merge.columns = colunas


    def valida_ean(self):
        import math
        ean_nao_encontrado = []
        for i, row in self.df_merge.iterrows():
            linha_embalagem = row['Embalagem']
            ean_cliente = row['Código']
            descricao = row['Descrição']
            validacao = self.qtd[self.qtd['="ean"'] == ean_cliente]

            print(i, ean_cliente)

            if "XXXX" in str(ean_cliente) or validacao.empty:
                print(f"Removendo o EAN: {ean_cliente}")
                validacao = self.qtd[self.qtd['="ean"'] == ean_cliente]
                ean_nao_encontrado.append(i)
                # print(f"Ean: {ean_cliente}")
                # print(f"Produto não encontrado: {descricao} não encontrado!")

        return ean_nao_encontrado


    def validar_quantidade(self):
    



        # Inicializar a lista de pedidos que nao atingem a qtd mínima
        lista_minimo = []

        # Iterar sobre cada linha do dataframe
        for i, row in self.df_merge.iterrows():
            quantidade_solicitada = None
            qtd_pedido = row['Qtd']
            embalagem_cliente = row['Embalagem']
            qtd_gam = row['="qt_embalagem_venda"']
                        
            if "nan" in str(embalagem_cliente) or "NaN" in str(embalagem_cliente):
                pass

            else:
                # Extrair o valor da embalagem do cliente
                embalagem_cliente_valor = int(embalagem_cliente.split(' x ')[1])
                embalagem_cliente_valor = int(embalagem_cliente_valor) * int(qtd_pedido)            

                # Ajustar quantidade_solicitada baseado na comparação
                if embalagem_cliente_valor < int(qtd_gam):
                    lista_minimo.append(i)
                    print(f"Quantidade do cliente menor que a quantidade da GAM, logo, não fechou o pedido mínimo.")
                    print(f"             ")

                # Se a quantidade que o cliente pediu for maior que o mínimo
                elif embalagem_cliente_valor > int(qtd_gam):
                    # Divide total cliente pela quantidade GAM e arredonda para baixo.
                    quantidade_solicitada = math.floor(embalagem_cliente_valor / int(qtd_gam))

                # Se a quantidade solicitada for igual...
                elif embalagem_cliente_valor == int(qtd_gam):
                    quantidade_solicitada = embalagem_cliente_valor

                # Atualizar a coluna 'Calculo Qtd Total' para a linha atual
                self.df_merge.at[i, 'Calculo Qtd Total'] = quantidade_solicitada


                print(f"Quantidade embalagem cliente: {embalagem_cliente}")
                print(f"Quantidade cliente: {qtd_pedido}")
                print(f"Quantidade GAM: {qtd_gam}")
                print(f"Quantidade solicitada será: {quantidade_solicitada}")

                print(f"             ")

        df_geral = self.df_merge.copy()

        # Remove pedidos que não atingiram o mínimo.
        dataframe_merge = self.df_merge.drop(lista_minimo)

        # Dataframes com pedidos que não atingiram o mínimo.
        df_minimo = df_geral.loc[lista_minimo]



        # retorna o dataframe atualizado
        return self.df_merge, df_minimo

In [ ]:
# INICIALIZAÇÃO E ETL
validacao = Validacoes(arqv, qtd)
validacao.processa_df()
validacao.etl_qtd()
validacao.join_colunas()

In [ ]:
# Execução - EAN

ean_n_encontrado = validacao.valida_ean()

In [ ]:
# Execução - Quantidade

df, minimo = validacao.validar_quantidade()